In [81]:
import pandas as pd
import numpy as np
import requests
import json
from bs4 import BeautifulSoup
from datetime import datetime
import time


In [82]:
# FPL Bootstrap Static
fpl_data = requests.get("https://fantasy.premierleague.com/api/bootstrap-static/").json()

players = pd.DataFrame(fpl_data['elements'])
teams = pd.DataFrame(fpl_data['teams'])
positions = pd.DataFrame(fpl_data['element_types'])

In [83]:
#Simplify players dataframe to relevant columns
players_df = players[['id', 'first_name', 'second_name', 'web_name', 'team','selected_by_percent','form' ,'now_cost',
                      'total_points','points_per_game', 'minutes', 'goals_scored', 'assists','goals_conceded', 'clean_sheets',
                      'penalties_saved','penalties_missed','yellow_cards','red_cards', 'saves','bonus','defensive_contribution_per_90','influence',
                      'creativity','threat','clearances_blocks_interceptions','defensive_contribution','starts','creativity_rank',
                      'threat_rank','saves_per_90','clean_sheets_per_90', 'goals_conceded_per_90','form_rank', 'expected_goals_per_90',
                      'expected_assists_per_90','expected_goal_involvements_per_90','element_type']]


#check dataframes
players_df.tail()


,id,first_name,second_name,web_name,team,selected_by_percent,form,now_cost,total_points,points_per_game,...,creativity_rank,threat_rank,saves_per_90,clean_sheets_per_90,goals_conceded_per_90,form_rank,expected_goals_per_90,expected_assists_per_90,expected_goal_involvements_per_90,element_type
740,663,Jhon,Arias,J.Arias,20,0.2,3.3,52,16,2.0,...,49,107,0.0,0.21,1.45,98,0.13,0.06,0.19,3
741,682,David,Møller Wolfe,Møller Wolfe,20,0.1,0.3,44,7,1.8,...,289,337,0.0,0.00,3.12,334,0.00,0.08,0.08,2
742,695,Jackson,Tchatchoua,Tchatchoua,20,0.0,2.0,45,10,1.4,...,116,257,0.0,0.00,2.20,162,0.06,0.01,0.07,2
743,709,Ladislav,Krejcí,Krejčí,20,0.1,1.0,45,11,2.2,...,134,82,0.0,0.00,1.60,248,0.15,0.06,0.21,2
744,720,Tolu,Arokodare,Tolu,20,0.1,1.0,55,7,1.4,...,292,109,0.0,0.00,3.07,266,0.17,0.02,0.19,4


In [84]:
#Simplify teams dataframe to relevant columns
teams_df = teams[['id', 'name', 'short_name', 'strength','played','win','loss','draw','points','form']]

#check dataframes
teams_df.head()

,id,name,short_name,strength,played,win,loss,draw,points,form
0,1,Arsenal,ARS,4,0,0,0,0,0,None
1,2,Aston Villa,AVL,3,0,0,0,0,0,None
2,3,Burnley,BUR,2,0,0,0,0,0,None
3,4,Bournemouth,BOU,3,0,0,0,0,0,None
4,5,Brentford,BRE,3,0,0,0,0,0,None


In [85]:
#Simplify positions dataframe to relevant columns
positions_df = positions[['id', 'singular_name', 'singular_name_short']]

#check dataframes
positions_df.head()

,id,singular_name,singular_name_short
0,1,Goalkeeper,GKP
1,2,Defender,DEF
2,3,Midfielder,MID
3,4,Forward,FWD


In [86]:
#Merge players with teams and positions
merged_df = players_df.merge(teams_df, left_on='team', right_on='id', suffixes=('_player', '_team'))
merged_df = merged_df.merge(positions_df, left_on='element_type', right_on='id', suffixes=('', '_position'))

#check merged dataframe
merged_df.head()

,id_player,first_name,second_name,web_name,team,selected_by_percent,form_player,now_cost,total_points,points_per_game,...,strength,played,win,loss,draw,points,form_team,id,singular_name,singular_name_short
0,1,David,Raya Martín,Raya,1,28.5,4.0,57,40,5.0,...,4,0,0,0,0,0,None,1,Goalkeeper,GKP
1,2,Kepa,Arrizabalaga Revuelta,Arrizabalaga,1,0.5,0.0,43,0,0.0,...,4,0,0,0,0,0,None,1,Goalkeeper,GKP
2,3,Karl,Hein,Hein,1,0.3,0.0,40,0,0.0,...,4,0,0,0,0,0,None,1,Goalkeeper,GKP
3,4,Tommy,Setford,Setford,1,0.2,0.0,40,0,0.0,...,4,0,0,0,0,0,None,1,Goalkeeper,GKP
4,5,Gabriel,dos Santos Magalhães,Gabriel,1,33.3,9.0,64,59,7.4,...,4,0,0,0,0,0,None,2,Defender,DEF


In [87]:
#list columns
merged_df.columns

Index(['id_player', 'first_name', 'second_name', 'web_name', 'team',
       'selected_by_percent', 'form_player', 'now_cost', 'total_points',
       'points_per_game', 'minutes', 'goals_scored', 'assists',
       'goals_conceded', 'clean_sheets', 'penalties_saved', 'penalties_missed',
       'yellow_cards', 'red_cards', 'saves', 'bonus',
       'defensive_contribution_per_90', 'influence', 'creativity', 'threat',
       'clearances_blocks_interceptions', 'defensive_contribution', 'starts',
       'creativity_rank', 'threat_rank', 'saves_per_90', 'clean_sheets_per_90',
       'goals_conceded_per_90', 'form_rank', 'expected_goals_per_90',
       'expected_assists_per_90', 'expected_goal_involvements_per_90',
       'element_type', 'id_team', 'name', 'short_name', 'strength', 'played',
       'win', 'loss', 'draw', 'points', 'form_team', 'id', 'singular_name',
       'singular_name_short'],
      dtype='object')

In [88]:
#remove columns not needed
final_df = merged_df.drop(columns=['web_name', 'id_team', 'element_type', 'short_name', 'strength', 'form_team', 'played',
                                    'win', 'loss', 'draw','points','id', 'singular_name_short','team'])

final_df.head()

,id_player,first_name,second_name,selected_by_percent,form_player,now_cost,total_points,points_per_game,minutes,goals_scored,...,threat_rank,saves_per_90,clean_sheets_per_90,goals_conceded_per_90,form_rank,expected_goals_per_90,expected_assists_per_90,expected_goal_involvements_per_90,name,singular_name
0,1,David,Raya Martín,28.5,4.0,57,40,5.0,720,0,...,729,1.88,0.62,0.38,69,0.00,0.00,0.0,Arsenal,Goalkeeper
1,2,Kepa,Arrizabalaga Revuelta,0.5,0.0,43,0,0.0,0,0,...,494,0.00,0.00,0.00,495,0.00,0.00,0.0,Arsenal,Goalkeeper
2,3,Karl,Hein,0.3,0.0,40,0,0.0,0,0,...,453,0.00,0.00,0.00,455,0.00,0.00,0.0,Arsenal,Goalkeeper
3,4,Tommy,Setford,0.2,0.0,40,0,0.0,0,0,...,475,0.00,0.00,0.00,478,0.00,0.00,0.0,Arsenal,Goalkeeper
4,5,Gabriel,dos Santos Magalhães,33.3,9.0,64,59,7.4,720,1,...,94,0.00,0.62,0.38,3,0.09,0.01,0.1,Arsenal,Defender


In [89]:
#list columns
final_df.columns

Index(['id_player', 'first_name', 'second_name', 'selected_by_percent',
       'form_player', 'now_cost', 'total_points', 'points_per_game', 'minutes',
       'goals_scored', 'assists', 'goals_conceded', 'clean_sheets',
       'penalties_saved', 'penalties_missed', 'yellow_cards', 'red_cards',
       'saves', 'bonus', 'defensive_contribution_per_90', 'influence',
       'creativity', 'threat', 'clearances_blocks_interceptions',
       'defensive_contribution', 'starts', 'creativity_rank', 'threat_rank',
       'saves_per_90', 'clean_sheets_per_90', 'goals_conceded_per_90',
       'form_rank', 'expected_goals_per_90', 'expected_assists_per_90',
       'expected_goal_involvements_per_90', 'name', 'singular_name'],
      dtype='object')

In [90]:
#rename columns for clarity
final_df = final_df.rename(columns={'singular_name': 'position','selected_by_percent': 'ownership (%)','id_player': 'player id', 
                                    'first_name': 'first name', 'second_name': 'last name', 'now_cost': 'cost (£m)',
                                    'total_points': 'total points','points_per_game': 'points per game', 'goals_scored': 'goals scored',
                                    'goals_conceded': 'goals conceded', 'clean_sheets': 'clean sheets',
                                    'penalties_saved': 'penalties saved','penalties_missed': 'penalties missed',
                                    'yellow_cards': 'yellow cards','red_cards': 'red cards', 'defensive_contribution_per_90': 'defensive contribution per 90',
                                    'defensive_contribution': 'defensive contribution','creativity_rank': 'creativity rank',
                                    'threat_rank': 'threat rank','form_rank': 'form rank',
                                    'expected_goals_per_90': 'xG per 90','expected_assists_per_90': 'xA per 90',
                                    'expected_goal_involvements_per_90': 'xGI per 90','saves_per_90': 'saves per 90',
                                    'clean_sheets_per_90': 'clean sheets per 90', 'goals_conceded_per_90': 'goals conceded per 90','name': 'Team',
                                    'form_player': 'form','minutes': 'minutes played','bonus': 'bonus points'})

final_df.head()

,player id,first name,last name,ownership (%),form,cost (£m),total points,points per game,minutes played,goals scored,...,threat rank,saves per 90,clean sheets per 90,goals conceded per 90,form rank,xG per 90,xA per 90,xGI per 90,Team,position
0,1,David,Raya Martín,28.5,4.0,57,40,5.0,720,0,...,729,1.88,0.62,0.38,69,0.00,0.00,0.0,Arsenal,Goalkeeper
1,2,Kepa,Arrizabalaga Revuelta,0.5,0.0,43,0,0.0,0,0,...,494,0.00,0.00,0.00,495,0.00,0.00,0.0,Arsenal,Goalkeeper
2,3,Karl,Hein,0.3,0.0,40,0,0.0,0,0,...,453,0.00,0.00,0.00,455,0.00,0.00,0.0,Arsenal,Goalkeeper
3,4,Tommy,Setford,0.2,0.0,40,0,0.0,0,0,...,475,0.00,0.00,0.00,478,0.00,0.00,0.0,Arsenal,Goalkeeper
4,5,Gabriel,dos Santos Magalhães,33.3,9.0,64,59,7.4,720,1,...,94,0.00,0.62,0.38,3,0.09,0.01,0.1,Arsenal,Defender


In [91]:
#check final dataframe columns
final_df.columns

Index(['player id', 'first name', 'last name', 'ownership (%)', 'form',
       'cost (£m)', 'total points', 'points per game', 'minutes played',
       'goals scored', 'assists', 'goals conceded', 'clean sheets',
       'penalties saved', 'penalties missed', 'yellow cards', 'red cards',
       'saves', 'bonus points', 'defensive contribution per 90', 'influence',
       'creativity', 'threat', 'clearances_blocks_interceptions',
       'defensive contribution', 'starts', 'creativity rank', 'threat rank',
       'saves per 90', 'clean sheets per 90', 'goals conceded per 90',
       'form rank', 'xG per 90', 'xA per 90', 'xGI per 90', 'Team',
       'position'],
      dtype='object')

In [92]:
# Function to reorder FPL DataFrame columns

def reorder_fpl_columns(final_df: pd.DataFrame) -> pd.DataFrame:
    """
    Reorders FPL DataFrame columns into logical categories:
    Identification → Ownership → Performance → Attack → Defense →
    Discipline → Goalkeeping → Metrics.
    
    Any columns not listed in the predefined order will be appended at the end.
    """
    
    # --- CATEGORY GROUPS ---
    identification = ['player id', 'first name', 'last name', 'Team', 'position']
    
    ownership_price = ['ownership (%)', 'cost (£m)', 'form', 'form rank']
    
    performance = ['total points', 'points per game', 'minutes played', 'starts']
    
    attack = ['goals scored', 'assists', 'xG per 90', 'xA per 90', 'xGI per 90']
    
    defense = ['clean sheets', 'goals conceded', 
               'defensive contribution', 'defensive contribution per 90', 
               'clearances_blocks_interceptions']
    
    discipline = ['yellow cards', 'red cards', 'penalties missed', 'penalties saved']
    
    goalkeeping = ['saves', 'saves per 90', 'clean sheets per 90', 'goals conceded per 90']
    
    metrics = ['influence', 'creativity', 'threat', 
               'creativity rank', 'threat rank', 'bonus points']
    
    # --- COMBINE ALL IN ORDER ---
    priority_order = (
        identification +
        ownership_price +
        performance +
        attack +
        defense +
        discipline +
        goalkeeping +
        metrics
    )
    
    # --- SAFELY REORDER ---
    ordered_cols = [col for col in priority_order if col in final_df.columns]
    remaining_cols = [col for col in final_df.columns if col not in ordered_cols]
    
    return final_df[ordered_cols + remaining_cols]


In [93]:
# Assuming 'df' is your combined FPL dataset
fpl_df = reorder_fpl_columns(final_df)

# Optional: view first few columns
print(fpl_df.columns[:15])


Index(['player id', 'first name', 'last name', 'Team', 'position',
       'ownership (%)', 'cost (£m)', 'form', 'form rank', 'total points',
       'points per game', 'minutes played', 'starts', 'goals scored',
       'assists'],
      dtype='object')


In [94]:
fpl_df.head()

,player id,first name,last name,Team,position,ownership (%),cost (£m),form,form rank,total points,...,saves,saves per 90,clean sheets per 90,goals conceded per 90,influence,creativity,threat,creativity rank,threat rank,bonus points
0,1,David,Raya Martín,Arsenal,Goalkeeper,28.5,57,4.0,69,40,...,15,1.88,0.62,0.38,140.4,10.0,0.0,330,729,3
1,2,Kepa,Arrizabalaga Revuelta,Arsenal,Goalkeeper,0.5,43,0.0,495,0,...,0,0.00,0.00,0.00,0.0,0.0,0.0,526,494,0
2,3,Karl,Hein,Arsenal,Goalkeeper,0.3,40,0.0,455,0,...,0,0.00,0.00,0.00,0.0,0.0,0.0,486,453,0
3,4,Tommy,Setford,Arsenal,Goalkeeper,0.2,40,0.0,478,0,...,0,0.00,0.00,0.00,0.0,0.0,0.0,507,475,0
4,5,Gabriel,dos Santos Magalhães,Arsenal,Defender,33.3,64,9.0,3,59,...,0,0.00,0.62,0.38,234.0,24.7,71.0,251,94,8


In [95]:
fpl_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 745 entries, 0 to 744
Data columns (total 37 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   player id                        745 non-null    int64  
 1   first name                       745 non-null    object 
 2   last name                        745 non-null    object 
 3   Team                             745 non-null    object 
 4   position                         745 non-null    object 
 5   ownership (%)                    745 non-null    object 
 6   cost (£m)                        745 non-null    int64  
 7   form                             745 non-null    object 
 8   form rank                        745 non-null    int64  
 9   total points                     745 non-null    int64  
 10  points per game                  745 non-null    object 
 11  minutes played                   745 non-null    int64  
 12  starts                

In [96]:
# --- Convert data types explicitly by column name ---

# 1️⃣ Convert "ownership (%)" → float
fpl_df["ownership (%)"] = (
    fpl_df["ownership (%)"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .astype(float)
)

# 2️⃣ Convert "cost (£m)" → float and divide by 10
fpl_df["cost (£m)"] = fpl_df["cost (£m)"].astype(float) / 10.0

# 3️⃣ Convert "form" and "points per game" → float
fpl_df["form"] = fpl_df["form"].astype(float)
fpl_df["points per game"] = fpl_df["points per game"].astype(float)

# 4️⃣ Convert "influence", "creativity", "threat" → integer
fpl_df["influence"] = fpl_df["influence"].astype(float).astype(int)
fpl_df["creativity"] = fpl_df["creativity"].astype(float).astype(int)
fpl_df["threat"] = fpl_df["threat"].astype(float).astype(int)

# ✅ Confirm updates
print(fpl_df.dtypes[["ownership (%)", "cost (£m)", "form", "points per game", 
                     "influence", "creativity", "threat"]])


ownership (%)      float64
cost (£m)          float64
form               float64
points per game    float64
influence            int64
creativity           int64
threat               int64
dtype: object


In [97]:
fpl_df.head(25)

,player id,first name,last name,Team,position,ownership (%),cost (£m),form,form rank,total points,...,saves,saves per 90,clean sheets per 90,goals conceded per 90,influence,creativity,threat,creativity rank,threat rank,bonus points
0,1,David,Raya Martín,Arsenal,Goalkeeper,28.5,5.7,4.0,69,40,...,15,1.88,0.62,0.38,140,10,0,330,729,3
1,2,Kepa,Arrizabalaga Revuelta,Arsenal,Goalkeeper,0.5,4.3,0.0,495,0,...,0,0.00,0.00,0.00,0,0,0,526,494,0
2,3,Karl,Hein,Arsenal,Goalkeeper,0.3,4.0,0.0,455,0,...,0,0.00,0.00,0.00,0,0,0,486,453,0
3,4,Tommy,Setford,Arsenal,Goalkeeper,0.2,4.0,0.0,478,0,...,0,0.00,0.00,0.00,0,0,0,507,475,0
4,5,Gabriel,dos Santos Magalhães,Arsenal,Defender,33.3,6.4,9.0,3,59,...,0,0.00,0.62,0.38,234,24,71,251,94,8
5,6,William,Saliba,Arsenal,Defender,12.6,6.0,3.8,75,31,...,0,0.00,0.72,0.18,125,22,10,263,298,1
6,7,Riccardo,Calafiori,Arsenal,Defender,14.9,5.8,3.8,74,48,...,0,0.00,0.73,0.44,118,74,127,108,41,3
7,8,Jurriën,Timber,Arsenal,Defender,20.8,5.9,5.0,37,54,...,0,0.00,0.59,0.44,207,154,189,35,14,7
8,9,Jakub,Kiwior,Arsenal,Defender,0.1,5.4,0.0,710,0,...,0,0.00,0.00,0.00,0,0,0,720,717,0
9,10,Myles,Lewis-Skelly,Arsenal,Defender,1.8,5.2,0.5,320,4,...,0,0.00,0.00,0.00,7,6,4,339,345,0
